# VectorBT A股小盘策略选型

> 已替换旧的“单标的双均线流程”。现在使用“小盘股池 + 三策略并行对比”主流程。

> 目标：避开大盘拥挤交易，利用小资金在小盘股中的灵活性。

In [4]:
import vectorbt as vbt
print('当前 vectorbt 版本:', vbt.__version__)

当前 vectorbt 版本: 0.28.5


## 1) 参数区（先改这里）

这一格集中管理新策略框架参数：
- 回测区间
- 交易成本（手续费/滑点）
- 小盘筛选阈值
- 三策略参数

In [5]:
# 1) 回测参数设置（新主流程）
start_date = '2025-06-01'
end_date = '2025-12-01'

init_cash = 100000
fees = 0.001
slippage = 0.001

# 运行控制参数
use_cache = True       # 使用本地缓存（推荐开启）
refresh_cache = True   # 首次建议 True：强制刷新一次后可改回 False
tushare_token = 'c2e997be7edf782f096c2b71a55cb1da4460a1aa6e6f2205'  # Notebook 内核读取不到系统环境变量时使用
tushare_http_url = 'http://teajoin.com'  # TeaJoin 文档入口
small_cap_quantile = 0.30   # 流通市值后30%
min_turnover = 5e7          # 日成交额下限
universe_size = 30          # 最多纳入标的数

# 策略A：趋势+移动止损
fast_window = 10
slow_window = 50
trail_stop = 0.08

# 策略B：小盘动量轮动
mom_window = 20
top_pct = 0.2

# 策略C：均值回归(RSI)
rsi_window = 14
rsi_buy = 30
rsi_sell = 55

print('当前参数设置：')
print(f'回测区间: {start_date} 到 {end_date}')
print(f'初始资金: {init_cash}')
print(f'手续费: {fees:.2%}, 滑点: {slippage:.2%}')
print(f'小盘阈值分位: {small_cap_quantile:.0%}, 成交额下限: {min_turnover:.0f}')
print(f'样本上限: {universe_size}')
print(f'启用缓存: {use_cache}, 强制刷新: {refresh_cache}')

当前参数设置：
回测区间: 2025-06-01 到 2025-12-01
初始资金: 100000
手续费: 0.10%, 滑点: 0.10%
小盘阈值分位: 30%, 成交额下限: 50000000
样本上限: 30
启用缓存: True, 强制刷新: True


In [6]:
# TuShare 官方写法连通性验证（TeaJoin 方式）
import tushare as ts

pro = ts.pro_api('c2e997be7edf782f096c2b71a55cb1da4460a1aa6e6f2205')
pro._DataApi__http_url = 'http://teajoin.com'

df_idx = pro.index_basic(limit=5)
print('index_basic 返回行数:', len(df_idx))

df_basic = pro.stock_basic(exchange='', list_status='L', fields='ts_code,symbol,name,market')
print('stock_basic 返回行数:', 0 if df_basic is None else len(df_basic))

ConnectionError: ('Connection aborted.', RemoteDisconnected('Remote end closed connection without response'))

## 2) 小盘股池构建

按流通市值和成交额筛选可交易小盘股，并拉取回测价格矩阵。
输出清单包含：代码、名称、PE、PB 等核心信息。

In [ ]:
# 2) 仅使用 TuShare 构建股票池、下载价格矩阵并做四因子打分
import os
import time
from pathlib import Path

import numpy as np
import pandas as pd
import tushare as ts

t0 = time.perf_counter()
cache_dir = Path('c:/Tarde/my_quant/outputs/cache')
cache_dir.mkdir(parents=True, exist_ok=True)
cache_file = cache_dir / f"smallcap_price_{start_date}_{end_date}_u{universe_size}.pkl"
name_map_file = cache_dir / f"smallcap_name_map_{start_date}_{end_date}_u{universe_size}.pkl"
metric_map_file = cache_dir / f"smallcap_metric_map_{start_date}_{end_date}_u{universe_size}.pkl"

def _to_yf_symbol(code6):
    code6 = str(code6).zfill(6)
    return f'{code6}.SS' if code6.startswith('6') else f'{code6}.SZ'

def _to_ts_code(symbol):
    code6 = str(symbol).split('.')[0].zfill(6)
    suffix = str(symbol).split('.')[-1].upper() if '.' in str(symbol) else ''
    exchange = 'SH' if suffix == 'SS' else 'SZ'
    return f'{code6}.{exchange}'

def _build_metric_map_from_ticker_df(df):
    out = {}
    for _, row in df.iterrows():
        ticker = str(row.get('ticker', ''))
        if not (ticker.endswith('.SS') or ticker.endswith('.SZ')):
            continue
        out[ticker] = {
            'PE': pd.to_numeric(row.get('PE', np.nan), errors='coerce'),
            'PB': pd.to_numeric(row.get('PB', np.nan), errors='coerce'),
            'ROE': pd.to_numeric(row.get('ROE', np.nan), errors='coerce'),
            'GROWTH': pd.to_numeric(row.get('GROWTH', np.nan), errors='coerce')
        }
    return out

def _name_map_is_valid(name_map_path):
    if not name_map_path.exists():
        return False
    try:
        name_map_cache = pd.read_pickle(name_map_path)
    except Exception:
        return False
    if not isinstance(name_map_cache, dict) or len(name_map_cache) == 0:
        return False
    valid = 0
    for value in name_map_cache.values():
        if pd.isna(value):
            continue
        name = str(value).strip().lower()
        if name and name not in ('nan', 'none'):
            valid += 1
    return valid >= max(3, int(len(name_map_cache) * 0.2))

def _ts_call_with_retry(callable_fn, max_retry=3, wait_sec=65):
    last_err = None
    for i in range(max_retry):
        try:
            return callable_fn()
        except Exception as e:
            last_err = e
            message = str(e)
            is_rate_limit = ('每分钟最多访问' in message) or ('频次' in message)
            if is_rate_limit and i < max_retry - 1:
                print(f'TuShare 限频，等待 {wait_sec}s 后重试 ({i + 2}/{max_retry})...')
                time.sleep(wait_sec)
                continue
            raise
    raise RuntimeError(f'TuShare 调用失败: {last_err}')

def _init_tushare_client():
    token = str(globals().get('tushare_token', '') or os.getenv('TUSHARE_TOKEN', '')).strip()
    if not token:
        raise RuntimeError('未检测到 TUSHARE_TOKEN 环境变量')

    ts.set_token(token)
    pro = ts.pro_api(token)
    http_url = str(globals().get('tushare_http_url', 'http://teajoin.com')).strip()
    if http_url:
        pro._DataApi__http_url = http_url
        print(f'已设置 TuShare HTTP 入口: {http_url}')
    return pro

def _get_last_trade_date_from_cal(pro, end_date_str):
    try:
        cal = _ts_call_with_retry(
            lambda: pro.trade_cal(exchange='SSE', start_date='20000101', end_date=end_date_str, fields='cal_date,is_open'),
            max_retry=2,
            wait_sec=2
        )
        if cal is not None and not cal.empty and 'cal_date' in cal.columns and 'is_open' in cal.columns:
            cal = cal[cal['is_open'] == 1]
            if not cal.empty:
                return str(cal['cal_date'].max())
    except Exception as e:
        print(f'trade_cal 不可用，使用回测结束日作为近似交易日: {e}')
    return end_date_str

def _safe_df_with_ts_code(df, columns):
    if df is None or (not isinstance(df, pd.DataFrame)) or df.empty:
        return pd.DataFrame(columns=columns)
    if 'ts_code' not in df.columns:
        return pd.DataFrame(columns=columns)
    for col in columns:
        if col not in df.columns:
            df[col] = np.nan
    return df[columns].copy()

def _fetch_pool_from_tushare(pro):
    end_str = pd.to_datetime(end_date).strftime('%Y%m%d')
    last_trade_date = _get_last_trade_date_from_cal(pro, end_str)

    daily = pd.DataFrame(columns=['ts_code', 'amount'])
    try:
        daily_try = _ts_call_with_retry(
            lambda: pro.daily(trade_date=last_trade_date, fields='ts_code,amount'),
            max_retry=2,
            wait_sec=3
        )
        daily = _safe_df_with_ts_code(daily_try, ['ts_code', 'amount'])
        if daily.empty:
            print('警告: TuShare daily(trade_date) 返回空，成交额筛选将降级。')
    except Exception as e:
        print(f'警告: TuShare daily(trade_date) 不可用，成交额筛选将降级: {e}')

    basic = _ts_call_with_retry(
        lambda: pro.stock_basic(exchange='', list_status='L', fields='ts_code,symbol,name,market')
    )
    basic = _safe_df_with_ts_code(basic, ['ts_code', 'symbol', 'name', 'market'])
    if basic.empty:
        raise RuntimeError('TuShare stock_basic 返回为空')

    basic['ticker'] = basic['ts_code'].astype(str).str.replace('.SH', '.SS', regex=False)
    basic['ticker'] = basic['ticker'].astype(str).str.replace('.SZ', '.SZ', regex=False)

    mv = pd.DataFrame(columns=['ts_code', 'circ_mv'])
    try:
        mv_try = _ts_call_with_retry(
            lambda: pro.daily_basic(trade_date=last_trade_date, fields='ts_code,circ_mv'),
            max_retry=2,
            wait_sec=3
        )
        mv = _safe_df_with_ts_code(mv_try, ['ts_code', 'circ_mv'])
        if mv.empty:
            print('警告: TuShare daily_basic(circ_mv) 返回空，小盘分位筛选将降级。')
    except Exception as e:
        print(f'警告: TuShare daily_basic(circ_mv) 不可用，小盘分位筛选将降级: {e}')

    pepb = pd.DataFrame(columns=['ts_code', 'pe_ttm', 'pb'])
    try:
        pepb_try = _ts_call_with_retry(
            lambda: pro.daily_basic(trade_date=last_trade_date, fields='ts_code,pe_ttm,pb'),
            max_retry=2,
            wait_sec=3
        )
        pepb = _safe_df_with_ts_code(pepb_try, ['ts_code', 'pe_ttm', 'pb'])
    except Exception:
        pass

    fina = pd.DataFrame(columns=['ts_code', 'roe'])
    try:
        fina_try = _ts_call_with_retry(
            lambda: pro.fina_indicator_vip(period=end_str, fields='ts_code,roe'),
            max_retry=2,
            wait_sec=3
        )
        fina = _safe_df_with_ts_code(fina_try, ['ts_code', 'roe'])
        if not fina.empty:
            fina = fina.drop_duplicates(subset=['ts_code'])
    except Exception:
        try:
            fina_try = _ts_call_with_retry(
                lambda: pro.fina_indicator(period=end_str, fields='ts_code,roe'),
                max_retry=2,
                wait_sec=3
            )
            fina = _safe_df_with_ts_code(fina_try, ['ts_code', 'roe'])
            if not fina.empty:
                fina = fina.drop_duplicates(subset=['ts_code'])
        except Exception:
            pass

    growth = pd.DataFrame(columns=['ts_code', 'or_yoy'])
    try:
        growth_try = _ts_call_with_retry(
            lambda: pro.fina_indicator_vip(period=end_str, fields='ts_code,or_yoy'),
            max_retry=2,
            wait_sec=3
        )
        growth = _safe_df_with_ts_code(growth_try, ['ts_code', 'or_yoy'])
        if not growth.empty:
            growth = growth.drop_duplicates(subset=['ts_code'])
    except Exception:
        try:
            growth_try = _ts_call_with_retry(
                lambda: pro.fina_indicator(period=end_str, fields='ts_code,or_yoy'),
                max_retry=2,
                wait_sec=3
            )
            growth = _safe_df_with_ts_code(growth_try, ['ts_code', 'or_yoy'])
            if not growth.empty:
                growth = growth.drop_duplicates(subset=['ts_code'])
        except Exception:
            pass

    pool = basic.copy()

    if not mv.empty:
        pool = pool.merge(mv, on='ts_code', how='left')
    else:
        pool['circ_mv'] = np.nan

    if not daily.empty:
        pool = pool.merge(daily, on='ts_code', how='left')
    else:
        pool['amount'] = np.nan

    if not pepb.empty:
        pool = pool.merge(pepb, on='ts_code', how='left')
    else:
        pool['pe_ttm'] = np.nan
        pool['pb'] = np.nan

    if not fina.empty:
        pool = pool.merge(fina, on='ts_code', how='left')
    else:
        pool['roe'] = np.nan

    if not growth.empty:
        pool = pool.merge(growth, on='ts_code', how='left')
    else:
        pool['or_yoy'] = np.nan

    pool['流通市值'] = pd.to_numeric(pool['circ_mv'], errors='coerce') * 1e4
    pool['成交额'] = pd.to_numeric(pool['amount'], errors='coerce') * 1e3
    pool['PE'] = pd.to_numeric(pool['pe_ttm'], errors='coerce')
    pool['PB'] = pd.to_numeric(pool['pb'], errors='coerce')
    pool['ROE'] = pd.to_numeric(pool['roe'], errors='coerce')
    pool['GROWTH'] = pd.to_numeric(pool['or_yoy'], errors='coerce')
    pool['名称'] = pool['name'].astype(str)

    out_cols = ['ticker', '名称', '流通市值', '成交额', 'PE', 'PB', 'ROE', 'GROWTH']
    for col in out_cols:
        if col not in pool.columns:
            pool[col] = np.nan
    pool = pool[out_cols].dropna(subset=['ticker'])

    return pool.sort_values(['流通市值', '成交额'], ascending=[True, False]).reset_index(drop=True)

def _fetch_price_history_from_tushare(pro, tickers, start, end):
    start_str = pd.to_datetime(start).strftime('%Y%m%d')
    end_str = pd.to_datetime(end).strftime('%Y%m%d')

    series_list = []
    for ticker in tickers:
        ts_code = _to_ts_code(ticker)
        hist_df = pd.DataFrame()
        try:
            hist_df = _ts_call_with_retry(
                lambda: ts.pro_bar(
                    ts_code=ts_code,
                    api=pro,
                    start_date=start_str,
                    end_date=end_str,
                    adj='qfq'
                ),
                max_retry=2,
                wait_sec=3
            )
        except Exception:
            try:
                hist_df = _ts_call_with_retry(
                    lambda: pro.daily(
                        ts_code=ts_code,
                        start_date=start_str,
                        end_date=end_str,
                        fields='trade_date,close'
                    ),
                    max_retry=2,
                    wait_sec=3
                )
            except Exception:
                hist_df = pd.DataFrame()

        if hist_df is None or hist_df.empty or ('trade_date' not in hist_df.columns) or ('close' not in hist_df.columns):
            continue

        price_series = pd.to_numeric(hist_df['close'], errors='coerce')
        price_series.index = pd.to_datetime(hist_df['trade_date'])
        price_series = price_series.sort_index().rename(ticker)
        price_series = price_series[~price_series.index.duplicated(keep='last')]
        series_list.append(price_series)

    if not series_list:
        return pd.DataFrame()

    return pd.concat(series_list, axis=1).sort_index()

if use_cache and (not refresh_cache) and cache_file.exists():
    has_token = bool((globals().get('tushare_token', '') or os.getenv('TUSHARE_TOKEN', '')).strip())
    if has_token and (not _name_map_is_valid(name_map_file)):
        print('检测到旧缓存名称字段缺失，自动触发一次刷新...')
        refresh_cache = True

if use_cache and (not refresh_cache) and cache_file.exists():
    price_df = pd.read_pickle(cache_file)
    name_map = pd.read_pickle(name_map_file) if name_map_file.exists() else {}
    metric_map = pd.read_pickle(metric_map_file) if metric_map_file.exists() else {}
    print(f'已加载本地缓存: {cache_file}')
    print(f'最终价格矩阵: {price_df.shape[0]} 行 x {price_df.shape[1]} 列')
    print(f'本格耗时: {time.perf_counter() - t0:.1f} 秒')
else:
    pro = _init_tushare_client()
    pool_df = _fetch_pool_from_tushare(pro)
    print('已使用 TuShare 构建股票池')

    cap_notna = pool_df['流通市值'].notna().sum()
    if cap_notna >= max(30, int(len(pool_df) * 0.1)):
        cap_threshold = pool_df['流通市值'].quantile(small_cap_quantile)
        pool_df = pool_df[pool_df['流通市值'] <= cap_threshold]
    else:
        print('警告: 流通市值字段不可用，已跳过小盘分位筛选，仅按成交额过滤。')

    turnover_notna = pool_df['成交额'].notna().sum()
    if turnover_notna >= max(30, int(len(pool_df) * 0.1)):
        pool_df = pool_df[pool_df['成交额'].fillna(0) >= min_turnover].copy()
    else:
        print('警告: TuShare 成交额字段不可用，已跳过成交额过滤。')

    pool_df = pool_df.drop_duplicates(subset=['ticker']).reset_index(drop=True)
    small_cap_universe = pool_df['ticker'].head(universe_size).tolist()
    name_map = dict(zip(pool_df['ticker'], pool_df['名称'].astype(str)))
    metric_map = _build_metric_map_from_ticker_df(pool_df)

    print(f'小盘候选数（截断前）: {len(pool_df)}')
    print(f'本次回测候选标的数: {len(small_cap_universe)}')
    print('样例标的:', small_cap_universe[:10])

    price_df = _fetch_price_history_from_tushare(pro, small_cap_universe, start_date, end_date)
    if price_df.empty:
        raise ValueError('TuShare 未返回有效价格矩阵，请检查 token 权限、网络或回测区间。')

    price_df = price_df.dropna(how='all').sort_index()
    price_df = price_df.ffill().dropna(axis=1, how='any')

    base_min_len = max(80, slow_window + 5)
    dynamic_cap = max(20, int(price_df.shape[0] * 0.8))
    min_len = min(base_min_len, dynamic_cap)
    valid_cols = [col for col in price_df.columns if price_df[col].notna().sum() >= min_len]
    price_df = price_df[valid_cols]

    if price_df.shape[1] < 10:
        raise ValueError('可回测标的不足10只，请放宽筛选条件、缩短区间或检查 TuShare 权限。')

    if use_cache:
        price_df.to_pickle(cache_file)
        pd.to_pickle(name_map, name_map_file)
        pd.to_pickle(metric_map, metric_map_file)
        print(f'已写入本地缓存: {cache_file}')

    print(f'最终价格矩阵: {price_df.shape[0]} 行 x {price_df.shape[1]} 列')
    print(f'本格耗时: {time.perf_counter() - t0:.1f} 秒')

if not isinstance(name_map, dict):
    name_map = {}
if not isinstance(metric_map, dict):
    metric_map = {}

# 输出样本清单（用于核查）
ticker_df = pd.DataFrame({'ticker': price_df.columns})
ticker_df['名称'] = ticker_df['ticker'].map(name_map).fillna('')

def _metric_value(tk, key):
    item = metric_map.get(tk, {})
    return item.get(key, np.nan)

ticker_df['PE'] = ticker_df['ticker'].apply(lambda x: _metric_value(x, 'PE'))
ticker_df['PB'] = ticker_df['ticker'].apply(lambda x: _metric_value(x, 'PB'))
ticker_df['ROE'] = ticker_df['ticker'].apply(lambda x: _metric_value(x, 'ROE'))
ticker_df['GROWTH'] = ticker_df['ticker'].apply(lambda x: _metric_value(x, 'GROWTH'))

out_dir = Path('c:/Tarde/my_quant/outputs')
out_dir.mkdir(parents=True, exist_ok=True)
ticker_df.to_csv(out_dir / '小盘池样本清单.csv', index=False, encoding='utf-8-sig')

print('股票池样本清单预览：')
display(ticker_df.head(15))
print(f'价格矩阵形状: {price_df.shape[0]} 行 x {price_df.shape[1]} 列')
print(f'本格总耗时: {time.perf_counter() - t0:.1f} 秒')

已设置 TuShare HTTP 入口: http://teajoin.com
警告: TuShare daily_basic(circ_mv) 返回空，小盘分位筛选将降级。
已使用 TuShare 构建股票池
警告: 流通市值字段不可用，已跳过小盘分位筛选，仅按成交额过滤。
小盘候选数（截断前）: 4330
本次回测候选标的数: 30
样例标的: ['300308.SZ', '000063.SZ', '300502.SZ', '601899.SS', '601138.SS', '300274.SZ', '300750.SZ', '688256.SS', '300476.SZ', '300456.SZ']
"None of [Index(['trade_date', 'adj_factor'], dtype='object')] are in the [columns]"
已写入本地缓存: c:\Tarde\my_quant\outputs\cache\smallcap_price_2025-06-01_2025-12-01_u30.pkl
最终价格矩阵: 124 行 x 29 列
本格耗时: 91.6 秒
股票池样本清单预览：


,ticker,名称,PE,PB,ROE,GROWTH
0,300308.SZ,中际旭创,69.9098,23.0404,NaN,NaN
1,000063.SZ,中兴通讯,37.9218,2.9544,NaN,NaN
2,300502.SZ,新易盛,47.0108,24.3367,NaN,NaN
3,601899.SS,紫金矿业,17.5368,4.7638,NaN,NaN
4,601138.SS,工业富联,38.8623,7.3530,NaN,NaN
5,300274.SZ,阳光电源,24.3026,8.6543,NaN,NaN
6,300750.SZ,宁德时代,27.4032,5.5598,NaN,NaN
7,688256.SS,寒武纪,306.1174,37.6382,NaN,NaN
8,300476.SZ,胜宏科技,64.3714,15.4217,NaN,NaN
9,300456.SZ,赛微电子,24.3607,5.4556,NaN,NaN


价格矩阵形状: 124 行 x 29 列
本格总耗时: 91.6 秒


In [ ]:
# 2b) 财务指标兜底补全与导出对象准备
# 如果部分股票的 ROE / GROWTH 在主流程里为空，这里直接按最近财报期逐级回查并写回。
# 这样后面的清单和导出都能拿到值，不再依赖旧缓存里的空字段。


def _financial_period_candidates(base_date):
    date_value = pd.Timestamp(base_date)
    candidates = [
        pd.Timestamp(date_value.year, 9, 30),
        pd.Timestamp(date_value.year, 6, 30),
        pd.Timestamp(date_value.year, 3, 31),
        pd.Timestamp(date_value.year - 1, 12, 31),
    ]
    return [period.strftime('%Y%m%d') for period in candidates if period <= date_value]

end_str = pd.to_datetime(end_date).strftime('%Y%m%d')
period_candidates = _financial_period_candidates(end_date)

if 'ticker_df' not in globals() or ticker_df.empty:
    ticker_df = pd.DataFrame({'ticker': price_df.columns if 'price_df' in globals() else []})

if 'name_map' not in globals() or not isinstance(name_map, dict):
    name_map = {}
if 'metric_map' not in globals() or not isinstance(metric_map, dict):
    metric_map = {}

missing_tickers = []
for ticker in ticker_df['ticker'].astype(str).tolist():
    metric = metric_map.get(ticker, {})
    roe_value = metric.get('ROE', np.nan)
    growth_value = metric.get('GROWTH', np.nan)
    if pd.isna(roe_value) or pd.isna(growth_value):
        missing_tickers.append(ticker)

if missing_tickers:
    print(f'发现 {len(missing_tickers)} 只标的缺少 ROE/GROWTH，开始兜底补全...')
    print('优先查询财报期:', ', '.join(period_candidates))
    for ticker in missing_tickers:
        ts_code = _to_ts_code(ticker)
        filled = False
        for period in period_candidates:
            try:
                fin = _ts_call_with_retry(
                    lambda: pro.fina_indicator(ts_code=ts_code, period=period, fields='ts_code,roe,or_yoy'),
                    max_retry=2,
                    wait_sec=2
                )
                if fin is not None and not fin.empty:
                    row = fin.iloc[0]
                    metric_map.setdefault(ticker, {})
                    if pd.notna(row.get('roe', np.nan)):
                        metric_map[ticker]['ROE'] = float(row['roe'])
                    if pd.notna(row.get('or_yoy', np.nan)):
                        metric_map[ticker]['GROWTH'] = float(row['or_yoy'])
                    if pd.notna(metric_map[ticker].get('ROE', np.nan)) or pd.notna(metric_map[ticker].get('GROWTH', np.nan)):
                        filled = True
                        break
            except Exception as e:
                last_err = e
                continue
        if not filled:
            print(f'兜底查询仍为空: {ticker}')
else:
    print('ROE/GROWTH 已完整，无需兜底补全。')

ticker_df['名称'] = ticker_df['ticker'].map(name_map).fillna('')
ticker_df['PE'] = ticker_df['ticker'].apply(lambda x: metric_map.get(x, {}).get('PE', np.nan))
ticker_df['PB'] = ticker_df['ticker'].apply(lambda x: metric_map.get(x, {}).get('PB', np.nan))
ticker_df['ROE'] = ticker_df['ticker'].apply(lambda x: metric_map.get(x, {}).get('ROE', np.nan))
ticker_df['GROWTH'] = ticker_df['ticker'].apply(lambda x: metric_map.get(x, {}).get('GROWTH', np.nan))

ticker_df = ticker_df.dropna(subset=['ticker']).reset_index(drop=True)
pool_output_df = ticker_df.copy()
print('财务指标补全完成，当前样本预览：')
display(ticker_df.head(15))

发现 29 只标的缺少 ROE/GROWTH，开始兜底补全...
优先查询财报期: 20250930, 20250630, 20250331, 20241231
兜底查询仍为空: 300102.SZ
财务指标补全完成，当前样本预览：


,ticker,名称,PE,PB,ROE,GROWTH
0,300308.SZ,中际旭创,69.9098,23.0404,31.3331,44.4312
1,000063.SZ,中兴通讯,37.9218,2.9544,7.1967,11.6332
2,300502.SZ,新易盛,47.0108,24.3367,55.3742,221.7035
3,601899.SS,紫金矿业,17.5368,4.7638,24.4994,10.3313
4,601138.SS,工业富联,38.8623,7.3530,14.3128,38.3979
5,300274.SZ,阳光电源,24.3026,8.6543,29.0246,32.9475
6,300750.SZ,宁德时代,27.4032,5.5598,17.4754,9.2753
7,688256.SS,寒武纪,306.1174,37.6382,19.1785,2386.3793
8,300476.SZ,胜宏科技,64.3714,15.4217,26.9289,83.3953
9,300456.SZ,赛微电子,24.3607,5.4556,26.8753,-17.3661


## 3) 三策略并行回测

并行比较趋势止损、动量轮动、RSI均值回归三类策略。

In [ ]:
# 3) 三策略并行回测与对比
strategy_rows = []
common_kwargs = dict(init_cash=init_cash, fees=fees, slippage=slippage, freq='1D')

# 策略A：趋势 + 移动止损
a_fast = vbt.MA.run(price_df, fast_window)
a_slow = vbt.MA.run(price_df, slow_window)
a_entries = a_fast.ma_crossed_above(a_slow)
a_exits = a_fast.ma_crossed_below(a_slow)
pf_a = vbt.Portfolio.from_signals(
    price_df, a_entries, a_exits, sl_stop=trail_stop, sl_trail=True, **common_kwargs
)
strategy_rows.append({
    '策略': '趋势+移动止损',
    '总收益率': float(pf_a.total_return().mean()),
    '最大回撤': float(pf_a.max_drawdown().mean()),
    '夏普比率': float(pf_a.sharpe_ratio().mean())
})

# 策略B：小盘动量轮动
mom = price_df.pct_change(mom_window)
rank = mom.rank(axis=1, ascending=False, method='first')
top_n = max(1, int(price_df.shape[1] * top_pct))
hold_mask = (rank <= top_n).astype(bool)
prev_hold = hold_mask.shift(1, fill_value=False).astype(bool)
b_entries = hold_mask & (~prev_hold)
b_exits = (~hold_mask) & prev_hold
pf_b = vbt.Portfolio.from_signals(price_df, b_entries, b_exits, **common_kwargs)
strategy_rows.append({
    '策略': '小盘动量轮动',
    '总收益率': float(pf_b.total_return().mean()),
    '最大回撤': float(pf_b.max_drawdown().mean()),
    '夏普比率': float(pf_b.sharpe_ratio().mean())
})

# 策略C：均值回归(RSI)
rsi = vbt.RSI.run(price_df, window=rsi_window).rsi
c_entries = rsi < rsi_buy
c_exits = rsi > rsi_sell
pf_c = vbt.Portfolio.from_signals(price_df, c_entries, c_exits, **common_kwargs)
strategy_rows.append({
    '策略': '均值回归(RSI)',
    '总收益率': float(pf_c.total_return().mean()),
    '最大回撤': float(pf_c.max_drawdown().mean()),
    '夏普比率': float(pf_c.sharpe_ratio().mean())
})

strategy_compare_df = pd.DataFrame(strategy_rows).sort_values('总收益率', ascending=False)
show_df = strategy_compare_df.copy()
show_df['总收益率'] = show_df['总收益率'].map(lambda x: f'{x:.2%}')
show_df['最大回撤'] = show_df['最大回撤'].map(lambda x: f'{x:.2%}')
show_df['夏普比率'] = show_df['夏普比率'].map(lambda x: f'{x:.4f}')

print('三策略对比（小盘池均值口径）：')
print(show_df.to_string(index=False))
show_df

三策略对比（小盘池均值口径）：
       策略   总收益率    最大回撤 夏普比率
   小盘动量轮动 32.99% -12.61%  inf
均值回归(RSI)  9.29%  -5.52%  inf
  趋势+移动止损  1.46%  -2.70%  inf


,策略,总收益率,最大回撤,夏普比率
1,小盘动量轮动,32.99%,-12.61%,inf
2,均值回归(RSI),9.29%,-5.52%,inf
0,趋势+移动止损,1.46%,-2.70%,inf


## 4) 结果解读

读取三策略对比结果，给出当前阶段的最优建议。

In [ ]:
# 4) 结果解读与策略选择建议
best_row = strategy_compare_df.iloc[0]
best_name = best_row['策略']
best_return = best_row['总收益率']
best_dd = best_row['最大回撤']
best_sharpe = best_row['夏普比率']

print('--- 当前最优策略建议 ---')
print(f"策略: {best_name}")
print(f"总收益率: {best_return:.2%}")
print(f"最大回撤: {best_dd:.2%}")
print(f"夏普比率: {best_sharpe:.4f}")

if abs(best_dd) <= 0.15:
    print('风险评估：满足你设定的回撤目标（<=15%）。')
else:
    print('风险评估：未达到回撤目标（>15%），建议继续调仓位或止损参数。')

--- 当前最优策略建议 ---
策略: 小盘动量轮动
总收益率: 32.99%
最大回撤: -12.61%
夏普比率: inf
风险评估：满足你设定的回撤目标（<=15%）。


## 5) 导出结果

导出策略对比结果，便于后续复盘与迭代。

In [ ]:
# 5) 导出策略对比结果
from pathlib import Path

output_dir = Path('c:/Tarde/my_quant/outputs')
output_dir.mkdir(parents=True, exist_ok=True)

compare_path = output_dir / '小盘三策略对比.csv'
strategy_compare_df.to_csv(compare_path, index=False, encoding='utf-8-sig')
print(f'已导出: {compare_path}')

pool_path = output_dir / '小盘池样本清单.csv'
ticker_df.to_csv(pool_path, index=False, encoding='utf-8-sig')
print(f'已导出: {pool_path}')

已导出: c:\Tarde\my_quant\outputs\小盘三策略对比.csv
已导出: c:\Tarde\my_quant\outputs\小盘池样本清单.csv
